## Synthèse de construction des datasets

### Dataset SFT

Le dataset SFT final est construit à partir de deux sources :

- **MediQAl** : corpus francophone orienté cas cliniques et raisonnement médical.
- **MedQuAD** : corpus anglophone de questions-réponses médicales générales.

Les exemples sont homogénéisés au format :

| Champ | Description |
|---|---|
| instruction | Question ou cas clinique présenté au modèle |
| response | Réponse attendue |
| source | Dataset d'origine |
| language | Langue de l'exemple |

### Prévention des fuites train/test

MediQAl contient plusieurs questions associées à un même cas clinique.  
Afin d'éviter qu'un même contexte patient apparaisse à la fois dans l'entraînement et dans l'évaluation, le split MediQAl est réalisé par groupe de `clinical_case`.

Les exemples sans `clinical_case` sont répartis aléatoirement, car aucun contexte patient partagé n'est identifiable.

### Nettoyage MedQuAD

L'analyse exploratoire a identifié 48 doublons exacts dans MedQuAD.  
Ces doublons ont été supprimés après homogénéisation au format SFT, en se basant sur le couple `instruction` / `response`.

### Dataset DPO

Le dataset DPO est construit à partir de **UltraMedical-Preference**.

Les exemples sont homogénéisés au format :

| Champ | Description |
|---|---|
| prompt | Question ou cas clinique |
| chosen_response | Réponse préférée |
| rejected_response | Réponse rejetée |
| source | Dataset d'origine |
| language | Langue |
| preference_type | Type de préférence (`easy`, `hard`, `length`) |

Le dataset DPO est divisé en trois jeux :

- train ;
- validation ;
- test.

### Traçabilité

Les transformations appliquées sont :

1. chargement des datasets publics Hugging Face ;
2. analyse exploratoire séparée des sources ;
3. homogénéisation au format SFT ou DPO ;
4. split groupé pour MediQAl afin d'éviter la fuite train/test ;
5. suppression des doublons exacts MedQuAD ;
6. échantillonnage contrôlé de MedQuAD et UltraMedical avec `seed=42` ;
7. sauvegarde des datasets finaux au format JSONL.

# Construction des datasets finaux

In [8]:
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict

## Chargement des datasets

In [9]:
mediqal = load_dataset("ANR-MALADES/MediQAl", "oeq")
medquad = load_dataset("keivalya/MedQuad-MedicalQnADataset")
ultra = load_dataset("TsinghuaC3I/UltraMedical-Preference")

print(mediqal)
print(medquad)
print(ultra)

DatasetDict({
    test: Dataset({
        features: ['id', 'clinical_case', 'cc_question_number', 'question', 'answer', 'medical_subject', 'question_type'],
        num_rows: 4969
    })
})
DatasetDict({
    train: Dataset({
        features: ['qtype', 'Question', 'Answer'],
        num_rows: 16407
    })
})
DatasetDict({
    train: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 109353
    })
    validation: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 2232
    })
    test: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 777
    })
})


## Préparation du dataset SFT — MediQAl

In [10]:
def format_mediqal(example):
    clinical_case = example["clinical_case"] or ""
    question = example["question"] or ""
    answer = example["answer"] or ""

    instruction = (clinical_case + "\n\nQuestion : " + question).strip()

    return {
        "instruction": instruction,
        "response": answer.strip(),
        "source": "MediQAl",
        "language": "fr"
    }

In [12]:
import random
from datasets import DatasetDict
#répartir MediQAl en train, validation, test sans séparer les questions d’un même cas clinique
SEED = 42

def split_mediqal_by_clinical_case(dataset, train_ratio=0.8, validation_ratio=0.1):
    random.seed(SEED)

    grouped_indices = {}
    no_case_indices = []

    for idx, example in enumerate(dataset):
        clinical_case = example["clinical_case"]

        if clinical_case is None or str(clinical_case).strip() == "":
            no_case_indices.append(idx)
        else:
            grouped_indices.setdefault(clinical_case, []).append(idx)

    clinical_cases = list(grouped_indices.keys())
    random.shuffle(clinical_cases)

    train_end = int(len(clinical_cases) * train_ratio)
    validation_end = int(len(clinical_cases) * (train_ratio + validation_ratio))

    train_cases = clinical_cases[:train_end]
    validation_cases = clinical_cases[train_end:validation_end]
    test_cases = clinical_cases[validation_end:]

    train_indices = [idx for case in train_cases for idx in grouped_indices[case]]
    validation_indices = [idx for case in validation_cases for idx in grouped_indices[case]]
    test_indices = [idx for case in test_cases for idx in grouped_indices[case]]

    random.shuffle(no_case_indices)

    no_case_train_end = int(len(no_case_indices) * train_ratio)
    no_case_validation_end = int(len(no_case_indices) * (train_ratio + validation_ratio))

    train_indices += no_case_indices[:no_case_train_end]
    validation_indices += no_case_indices[no_case_train_end:no_case_validation_end]
    test_indices += no_case_indices[no_case_validation_end:]

    return DatasetDict({
        "train": dataset.select(train_indices),
        "validation": dataset.select(validation_indices),
        "test": dataset.select(test_indices),
    })

In [13]:
mediqal_split = split_mediqal_by_clinical_case(mediqal["test"])

mediqal_split

DatasetDict({
    train: Dataset({
        features: ['id', 'clinical_case', 'cc_question_number', 'question', 'answer', 'medical_subject', 'question_type'],
        num_rows: 3975
    })
    validation: Dataset({
        features: ['id', 'clinical_case', 'cc_question_number', 'question', 'answer', 'medical_subject', 'question_type'],
        num_rows: 518
    })
    test: Dataset({
        features: ['id', 'clinical_case', 'cc_question_number', 'question', 'answer', 'medical_subject', 'question_type'],
        num_rows: 476
    })
})

In [14]:
#formatage
columns_to_remove_mediqal = [
    "id",
    "clinical_case",
    "cc_question_number",
    "question",
    "answer",
    "medical_subject",
    "question_type"
]

mediqal_sft = DatasetDict({
    split_name: mediqal_split[split_name]
        .map(format_mediqal)
        .remove_columns(columns_to_remove_mediqal)
    for split_name in ["train", "validation", "test"]
})

mediqal_sft

Map:   0%|          | 0/3975 [00:00<?, ? examples/s]

Map:   0%|          | 0/518 [00:00<?, ? examples/s]

Map:   0%|          | 0/476 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'source', 'language'],
        num_rows: 3975
    })
    validation: Dataset({
        features: ['instruction', 'response', 'source', 'language'],
        num_rows: 518
    })
    test: Dataset({
        features: ['instruction', 'response', 'source', 'language'],
        num_rows: 476
    })
})

## Préparation du dataset SFT — MedQuAD

In [15]:
medquad_sft = medquad["train"].map(lambda x: {
    "instruction": (x["Question"] or "").strip(),
    "response": (x["Answer"] or "").strip(),
    "source": "MedQuAD",
    "language": "en"
})

medquad_sft = medquad_sft.remove_columns([
    "qtype",
    "Question",
    "Answer"
])

medquad_sft[0]

{'instruction': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?',
 'response': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.',
 'source': 'MedQuAD',
 'language': 'en'}

In [ ]:
# Suppression des doublons
medquad_sft = medquad["train"].map(lambda x: {
    "instruction": (x["Question"] or "").strip(),
    "response": (x["Answer"] or "").strip(),
    "source": "MedQuAD",
    "language": "en"
})

medquad_sft = medquad_sft.remove_columns([
    "qtype",
    "Question",
    "Answer"
])

print("Avant suppression des doublons :", len(medquad_sft))

medquad_df = medquad_sft.to_pandas()

medquad_df = medquad_df.drop_duplicates(
    subset=["instruction", "response"]
).reset_index(drop=True)

medquad_sft = Dataset.from_pandas(
    medquad_df,
    preserve_index=False
)

print("Après suppression des doublons :", len(medquad_sft))

medquad_sft[0]

Avant suppression des doublons : 16407
Après suppression des doublons : 16359


{'instruction': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?',
 'response': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.',
 'source': 'MedQuAD',
 'language': 'en'}

## Échantillonnage du dataset SFT

In [19]:
medquad_sample = medquad_sft.shuffle(seed=42).select(range(1000))

medquad_split = medquad_sample.train_test_split(
    test_size=0.2,
    seed=42
)

medquad_temp_split = medquad_split["test"].train_test_split(
    test_size=0.5,
    seed=42
)

medquad_sft_split = DatasetDict({
    "train": medquad_split["train"],
    "validation": medquad_temp_split["train"],
    "test": medquad_temp_split["test"]
})

medquad_sft_split

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'source', 'language'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['instruction', 'response', 'source', 'language'],
        num_rows: 100
    })
    test: Dataset({
        features: ['instruction', 'response', 'source', 'language'],
        num_rows: 100
    })
})

## Fusion des datasets

In [21]:
# Mise au meme format pour les deux datasets
from datasets import Features, Value

sft_features = Features({
    "instruction": Value("string"),
    "response": Value("string"),
    "source": Value("string"),
    "language": Value("string"),
})

mediqal_sft = DatasetDict({
    split_name: mediqal_sft[split_name].cast(sft_features)
    for split_name in ["train", "validation", "test"]
})

medquad_sft_split = DatasetDict({
    split_name: medquad_sft_split[split_name].cast(sft_features)
    for split_name in ["train", "validation", "test"]
})

Casting the dataset:   0%|          | 0/3975 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/518 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/476 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [23]:
sft_final = DatasetDict({
    "train": concatenate_datasets([mediqal_sft["train"], medquad_sft_split["train"]]),
    "validation": concatenate_datasets([mediqal_sft["validation"], medquad_sft_split["validation"]]),
    "test": concatenate_datasets([mediqal_sft["test"], medquad_sft_split["test"]]),
})
sft_final

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'source', 'language'],
        num_rows: 4775
    })
    validation: Dataset({
        features: ['instruction', 'response', 'source', 'language'],
        num_rows: 618
    })
    test: Dataset({
        features: ['instruction', 'response', 'source', 'language'],
        num_rows: 576
    })
})

In [24]:
sft_final["train"].to_json("../../data/sft_train.jsonl")
sft_final["validation"].to_json("../../data/sft_validation.jsonl")
sft_final["test"].to_json("../../data/sft_test.jsonl")

print("SFT train :", len(sft_final["train"]))
print("SFT validation :", len(sft_final["validation"]))
print("SFT test :", len(sft_final["test"]))

Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

SFT train : 4775
SFT validation : 618
SFT test : 576


## Préparation du dataset DPO — UltraMedical

In [25]:
ultra_sample = ultra["train"].shuffle(seed=42).select(range(2000))

def extract_assistant_content(messages):
    for message in reversed(messages):
        if message.get("role") == "assistant":
            return message.get("content", "").strip()
    return messages[-1].get("content", "").strip()

def format_dpo(example):
    return {
        "prompt": example["prompt"].strip(),
        "chosen_response": extract_assistant_content(example["chosen"]),
        "rejected_response": extract_assistant_content(example["rejected"]),
        "source": "UltraMedical-Preference",
        "language": "en",
        "preference_type": example["label_type"]
    }

dpo_dataset = ultra_sample.map(format_dpo)

dpo_dataset = dpo_dataset.remove_columns([
    "prompt_id",
    "label_type",
    "chosen",
    "rejected",
    "metadata",
    "feedback"
])

dpo_dataset[0]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'prompt': 'In a 50-year-old man with type 2 diabetes mellitus on metformin, who presents with an acutely painful, red, and hard right eye, along with a frontal headache and gastrointestinal symptoms such as nausea with vomiting, which condition would be most consistent with these findings, especially when considering the photophobia, blurred vision, and a mid-dilated, nonreactive right pupil, but also factoring in his diabetes management history?\n\nA. Acute angle-closure glaucoma\nB. Orbital cellulitis with ocular muscle involvement\nC. Migraine with aura and ocular involvement\nD. Diabetic retinopathy with neovascularization',
 'chosen_response': "The symptoms presented by the patient - an acutely painful, red, and hard eye, frontal headache, nausea with vomiting, photophobia, blurred vision, and a mid-dilated, nonreactive pupil - are classic signs of acute angle-closure glaucoma. Acute angle-closure glaucoma is an ophthalmic emergency characterized by a sudden increase in intraocul

## Split du fichier DPO

In [27]:
dpo_split = dpo_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

dpo_temp_split = dpo_split["test"].train_test_split(
    test_size=0.5,
    seed=42
)

dpo_final = DatasetDict({
    "train": dpo_split["train"],
    "validation": dpo_temp_split["train"],
    "test": dpo_temp_split["test"]
})

dpo_final

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen_response', 'rejected_response', 'source', 'language', 'preference_type'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['prompt', 'chosen_response', 'rejected_response', 'source', 'language', 'preference_type'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'chosen_response', 'rejected_response', 'source', 'language', 'preference_type'],
        num_rows: 200
    })
})

## Sauvegarde du dataset DPO

In [28]:
dpo_final["train"].to_json("../../data/dpo_train.jsonl")
dpo_final["validation"].to_json("../../data/dpo_validation.jsonl")
dpo_final["test"].to_json("../../data/dpo_test.jsonl")

# Optionnel : garder aussi le dataset complet pour compatibilité
dpo_dataset.to_json("../../data/dpo_dataset.jsonl")

print("DPO train :", len(dpo_final["train"]))
print("DPO validation :", len(dpo_final["validation"]))
print("DPO test :", len(dpo_final["test"]))

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

DPO train : 1600
DPO validation : 200
DPO test : 200


In [29]:
print("Exemple SFT train")
print(sft_final["train"][0])

print("\nExemple SFT validation")
print(sft_final["validation"][0])

print("\nExemple DPO train")
print(dpo_final["train"][0])

print("\nExemple DPO validation")
print(dpo_final["validation"][0])

Exemple SFT train
{'instruction': 'Monsieur D., 77 ans, est hospitalisé à la demande de son médecin traitant pour\nconfusion aiguë.\nDans ses antécédents, figurent un infarctus du myocarde (qu’il a présenté il y a 10\nans), une artériopathie oblitérante des membres inférieurs, une hypertension\nartérielle et un diabète depuis 20 ans devenu insulino requérant depuis 10 ans.\nC’était un gros fumeur, sevré depuis une dizaine d’années.\nSon traitement habituel est le suivant :\n\nUMULINE® NPH (16 U le matin, 16 U le soir)\nACTOS® 15 (pioglitazone) : 1cp/j\nELISOR® 40 (pravastatine) : 1cp/j\nLASILIX® FAIBLE 20 mg (furosémide) : 1cp/j\nRENITEC® 20 (énalapril) : 1cp/j\nPLAVIX® 75 (clopidogrel) : 1cp/j\nNITRIDERM TTS® 5 mg (trinitrine) : 1/j\n\nA l’examen clinique d’entrée il est confus, très dyspnéique, encombré. Il présente de\ndiscrets oedèmes remontant jusqu’aux cuisses, sans reflux hépatojugulaire. La toux\nramène une expectoration mousseuse. Il présente une fi

## Schéma des métadonnées

## Schéma des métadonnées

### Dataset SFT

| Champ | Description |
|---------|---------|
| instruction | Question ou cas clinique présenté au modèle |
| response | Réponse attendue |
| source | Dataset d'origine (MediQAl ou MedQuAD) |
| language | Langue de l'exemple (fr ou en) |

### Dataset DPO

| Champ | Description |
|---------|---------|
| prompt | Question ou cas clinique |
| chosen_response | Réponse préférée par les annotateurs |
| rejected_response | Réponse moins pertinente |
| source | Dataset d'origine |
| language | Langue de l'exemple |
| preference_type | Type de préférence issu d'UltraMedical (`easy`, `hard`, `length`) |

### Métadonnées médicales

Les datasets sources ne fournissent pas systématiquement des informations structurées concernant :

- les symptômes ;
- les antécédents ;
- les constantes vitales ;
- le niveau de confiance.

Ces informations peuvent être présentes dans le texte des cas cliniques, mais ne sont pas disponibles sous forme de champs dédiés homogènes entre les sources. Elles ne sont donc pas intégrées comme colonnes structurées dans le dataset final.

En revanche, les métadonnées minimales `source` et `language` sont conservées pour assurer la traçabilité de chaque exemple. Pour le DPO, le champ `preference_type` est également conservé afin de documenter la nature de la préférence annotée.

## Considérations RGPD et anonymisation

Les datasets utilisés proviennent exclusivement de sources publiques diffusées à des fins de recherche ou d'enseignement :

- MediQAl ;
- MedQuAD ;
- UltraMedical-Preference.

Aucune donnée clinique réelle issue du Centre Hospitalier Saint-Aurélien ou d'un établissement de santé n'a été utilisée dans ce POC.

Une analyse exploratoire de la structure des données a été réalisée. Aucun champ explicitement dédié à des informations personnelles identifiantes n'a été identifié, notamment :

- nom ou prénom de patient ;
- adresse ;
- numéro de téléphone ;
- email ;
- identifiant patient ;
- numéro de sécurité sociale.

Les informations potentiellement sensibles peuvent toutefois apparaître dans le texte libre des cas cliniques. Pour cette raison, les données sont considérées comme des données médicales publiques à manipuler avec prudence.

L'utilisation d'un outil d'anonymisation automatique tel que Presidio a été étudiée. Cependant, dans ce projet, son usage systématique n'a pas été retenu, car les textes médicaux peuvent contenir de nombreux faux positifs : noms de maladies, syndromes, médicaments, éponymes médicaux ou noms de services pouvant être confondus avec des entités personnelles.

Mesures retenues pour le POC :

1. utilisation exclusive de datasets publics ;
2. conservation des sources via le champ `source` ;
3. absence d'ajout de données patient réelles ;
4. limitation des métadonnées aux champs nécessaires ;
5. séparation des jeux train / validation / test ;
6. documentation des transformations appliquées dans le notebook.

Pour un passage en production ou l'intégration de données hospitalières réelles, une anonymisation formelle serait obligatoire avec validation humaine, contrôle qualité et traçabilité des opérations de masquage.